# RAG Complete Module : LangChain 2026
### Full Pipeline: LLM → Prompt → Output Parser → Loader → Splitter → Embeddings → VectorStore → Retriever → RAG

---

| # | Section |
|---|-----------------------------|
| 1 | Groq LLM Setup              |
| 2 | Prompt Template (LCEL)      |
| 3 | Output Parsers              |
| 4 | Document Loaders            |
| 5 | Text Splitters              |
| 6 | Embeddings                  |
| 7 | Vector Store (FAISS)        |
| 8 | Retriever                   |
| 9 | Full RAG Chain (LCEL)       |

---


## API

- We often need to use **API keys** (like Groq API Key) to connect with external services.  
- Instead of writing the key directly in the code, we store it safely in a hidden file called `.env`.  
- The `dotenv` library helps us **load values from the `.env` file** into our program.  
- `os.getenv("GROQ_API_KEY")` reads the key from environment variables.  
- This way, the code stays clean, secure, and reusable without exposing sensitive information.  
- The print statement checks if the key was loaded successfully:
  - Key found → ready to use  
  - Key missing → you need to set it in `.env` or directly in the code


In [13]:
import os
from dotenv import load_dotenv
load_dotenv()

# For classroom demo — set directly here
# os.environ["GROQ_API_KEY"] = "gsk_xxxxxxxxxxxx"

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print("✅ Groq API Key Loaded" if GROQ_API_KEY else "❌ Set GROQ_API_KEY")

✅ Groq API Key Loaded


In [14]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── LLM ──
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    max_tokens=1024,
    api_key=GROQ_API_KEY
)

# ── Simple Prompt ──
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a cricket analyst. Answer based on cricket match data."),
    ("human", "{question}")
])

# ── RAG Prompt — {context} filled by Retriever, {question} from user ──
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are a cricket analyst assistant.
Use ONLY the following context to answer the question.
If the answer is not in the context, say 'I do not have enough data.'

Context:
{context}"""),
    ("human", "{question}")
])

str_parser  = StrOutputParser()
json_parser = JsonOutputParser()

print("LLM + Prompts + Parsers — reloaded")

LLM + Prompts + Parsers — reloaded


In [15]:
from langchain_community.document_loaders import TextLoader, WebBaseLoader, JSONLoader
from langchain_community.document_loaders.csv_loader import CSVLoader
import bs4

txt_docs = TextLoader(
    r"C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\match_summary.txt",
    encoding="utf-8"
).load()

batting_docs = CSVLoader(
    r"C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\batting_scorecard.csv",
    encoding="utf-8"
).load()

bowling_docs = CSVLoader(
    r"C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\bowling_figures.csv",
    encoding="utf-8"
).load()

json_docs = JSONLoader(
    file_path=r"C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\input.json",
    jq_schema=".",
    text_content=False
).load()

web_docs = WebBaseLoader(
    web_paths=["https://www.crictracker.com/t20/ipl-indian-premier-league/points-table/?ref=hm"],
    bs_kwargs={"parse_only": bs4.SoupStrainer(
        class_=["entry-content", "standings", "points-table", "table", "main-content"]
    )},
    header_template={"User-Agent": "Mozilla/5.0"}
).load()

all_docs = txt_docs + batting_docs + bowling_docs + json_docs + web_docs

print(f"Documents reloaded : {len(all_docs)} total")
print(f"   TXT={len(txt_docs)} | Batting={len(batting_docs)} | Bowling={len(bowling_docs)} | JSON={len(json_docs)} | Web={len(web_docs)}")

Documents reloaded : 27 total
   TXT=1 | Batting=13 | Bowling=11 | JSON=1 | Web=1


### RecursiveCharacterTextSplitter *(Best Choice for RAG)*

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [17]:
final_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    add_start_index=True
)

all_chunks = final_splitter.split_documents(all_docs)

print(f"all_chunks = {len(all_chunks)} chunks")
print(f"These are the INPUT for Section 6 Embeddings")

all_chunks = 46 chunks
These are the INPUT for Section 6 Embeddings


In [18]:
from langchain_huggingface import HuggingFaceEmbeddings

# Downloads ~90MB on first run — then cached locally
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}   # normalise for accurate cosine similarity
)

print("Embedding model loaded")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3221.62it/s]


Embedding model loaded


In [19]:
# ── Similarity Demo — prove same meaning = similar vectors ──
import numpy as np

def cosine_similarity(v1, v2):
    v1, v2 = np.array(v1), np.array(v2)
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

sentences = {
    "A": "Rohit Sharma hit a century in IPL",
    "B": "The batsman scored 100 runs in the tournament",   # similar meaning
    "C": "The pitch was wet due to heavy rain",              # different topic
}

vecs = {k: embeddings.embed_query(v) for k, v in sentences.items()}

print("A :", sentences["A"])
print("B :", sentences["B"])
print("C :", sentences["C"])
print()
print(f"Similarity A ↔ B (same meaning) : {cosine_similarity(vecs['A'], vecs['B']):.4f}")
print(f"Similarity A ↔ C (diff meaning) : {cosine_similarity(vecs['A'], vecs['C']):.4f}")
print("\n A↔B score MUCH higher than A↔C — this is how semantic search works!")

A : Rohit Sharma hit a century in IPL
B : The batsman scored 100 runs in the tournament
C : The pitch was wet due to heavy rain

Similarity A ↔ B (same meaning) : 0.4467
Similarity A ↔ C (diff meaning) : 0.1604

 A↔B score MUCH higher than A↔C — this is how semantic search works!


In [20]:
from langchain_community.vectorstores import FAISS

# Embeds every chunk and stores (text, vector) pairs in the FAISS index
print("Building vector store — embedding all chunks (30-60 sec)...")

vectorstore = FAISS.from_documents(
    documents=all_chunks,
    embedding=embeddings
)

print(f"Vector Store ready!")
print(f"   Vectors stored : {vectorstore.index.ntotal}")

Building vector store — embedding all chunks (30-60 sec)...
Vector Store ready!
   Vectors stored : 46


In [21]:
# ── Similarity Search — raw demo ──
query = "Who took the most wickets?"
results = vectorstore.similarity_search(query, k=3)

print(f"Query: '{query}'")
print(f"Top {len(results)} matching chunks:\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source  : {doc.metadata.get('source', 'unknown')}")
    print(f"Content : {doc.page_content[:200]}")
    print()

Query: 'Who took the most wickets?'
Top 3 matching chunks:

--- Result 1 ---
Source  : C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\match_summary.txt
Content : Fall of Wickets (DC):
1st wicket: 29 runs (Nissanka, over 3.6)
2nd wicket: 36 runs (KL Rahul, over 5.1)
3rd wicket: 52 runs (Karun Nair, over 7.6)
4th wicket: 61 runs (Nitish Rana, over 9.3)
5th wicke

--- Result 2 ---
Source  : C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\match_summary.txt
Content : Fall of Wickets (CSK):
1st wicket: 24 runs (Ruturaj Gaikwad, over 3.5)
2nd wicket: 45 runs (Urvil Patel, over 6.3)

--- Result 3 ---
Source  : C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\match_summary.txt
Content : === CSK BOWLING (vs DC) ===
Akeal Hosein: 4 overs, 0 maiden, 19 runs, 1 wicket, Econ: 4.75
Mukesh Choudhary: 4 overs, 0 maiden, 31 runs, 1 wicket, Econ: 7.75
Anshul Kamboj: 4 overs, 0 maiden, 49 runs,



In [22]:
# ── With Score — shows how similar each result is ──
results_scored = vectorstore.similarity_search_with_score(query, k=3)

print(f"Query: '{query}' — with scores (lower = more similar in FAISS L2)\n")
for doc, score in results_scored:
    print(f"Score   : {score:.4f}")
    print(f"Content : {doc.page_content[:150]}")
    print()

Query: 'Who took the most wickets?' — with scores (lower = more similar in FAISS L2)

Score   : 0.7286
Content : Fall of Wickets (DC):
1st wicket: 29 runs (Nissanka, over 3.6)
2nd wicket: 36 runs (KL Rahul, over 5.1)
3rd wicket: 52 runs (Karun Nair, over 7.6)
4th

Score   : 0.7857
Content : Fall of Wickets (CSK):
1st wicket: 24 runs (Ruturaj Gaikwad, over 3.5)
2nd wicket: 45 runs (Urvil Patel, over 6.3)

Score   : 0.7953
Content : === CSK BOWLING (vs DC) ===
Akeal Hosein: 4 overs, 0 maiden, 19 runs, 1 wicket, Econ: 4.75
Mukesh Choudhary: 4 overs, 0 maiden, 31 runs, 1 wicket, Eco



In [23]:
# ── Save & Load FAISS index (production pattern) ──
FAISS_PATH = r"C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\faiss_index"

# Save
vectorstore.save_local(FAISS_PATH)
print(f"Saved to: {FAISS_PATH}")

# Load (next time no need to re-embed everything)
vectorstore_loaded = FAISS.load_local(
    FAISS_PATH,
    embeddings,
    allow_dangerous_deserialization=True   # Required in LangChain 2026
)
print(f"Loaded from disk — vectors: {vectorstore_loaded.index.ntotal}")

Saved to: C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\faiss_index
Loaded from disk — vectors: 46


In [24]:
# ── Basic Retriever ──
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}   # Return top 4 most relevant chunks
)

# Test — retriever is a Runnable so use .invoke()
retrieved = retriever.invoke("Which team is leading IPL?")

print(f"Retrieved {len(retrieved)} chunks:\n")
for i, doc in enumerate(retrieved):
    print(f"[{i+1}] {doc.page_content[:200]}")
    print()

Retrieved 4 chunks:

[1] IPL 2026 - Match 48 (Night Game)
Venue: Delhi, May 05, 2026
Indian Premier League

=== RESULT ===
Chennai Super Kings beat Delhi Capitals by 8 wickets (with 15 balls remaining)
Delhi Capitals scored 1

[2] match: IPL2026_M48
innings: 1st innings
team_bowling: Chennai Super Kings
bowler: Noor Ahmad
overs: 3
maidens: 0
runs: 22
wickets: 2
economy: 7.33
dot_balls: 6
wides: 0
no_balls: 0

[3] match: IPL2026_M48
innings: 1st innings
team_bowling: Chennai Super Kings
bowler: Jamie Overton
overs: 1
maidens: 0
runs: 5
wickets: 1
economy: 5.00
dot_balls: 1
wides: 0
no_balls: 0

[4] match: IPL2026_M48
innings: 2nd innings
team_bowling: Delhi Capitals
bowler: Axar Patel
overs: 4
maidens: 0
runs: 25
wickets: 1
economy: 6.25
dot_balls: 12
wides: 1
no_balls: 0



In [32]:
retrieved

[Document(id='a2f9d8c5-a604-4e37-8606-174b8cbf2eb5', metadata={'source': 'C:\\Users\\admin\\Desktop\\New_GenAI\\GenAI\\Langchain\\Doc_loader\\match_summary.txt', 'start_index': 0}, page_content='IPL 2026 - Match 48 (Night Game)\nVenue: Delhi, May 05, 2026\nIndian Premier League\n\n=== RESULT ===\nChennai Super Kings beat Delhi Capitals by 8 wickets (with 15 balls remaining)\nDelhi Capitals scored 155/7 in 20 overs.\nChennai Super Kings chased it down in 17.3 overs, finishing at 159/2.\n\n=== PLAYER OF THE MATCH ===\nSanju Samson (CSK) - 87* off 52 balls (7 fours, 6 sixes, SR: 167.30)\nCricinfo MVP Points: 105.96'),
 Document(id='1cfd812a-acce-4855-b28f-ecca857e13ff', metadata={'source': 'C:\\Users\\admin\\Desktop\\New_GenAI\\GenAI\\Langchain\\Doc_loader\\bowling_figures.csv', 'row': 3, 'start_index': 0}, page_content='match: IPL2026_M48\ninnings: 1st innings\nteam_bowling: Chennai Super Kings\nbowler: Noor Ahmad\novers: 3\nmaidens: 0\nruns: 22\nwickets: 2\neconomy: 7.33\ndot_balls: 6\n

In [25]:
# ── MMR Retriever — avoids near-duplicate chunks ──
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 10}   # fetch 10, keep most diverse 4
)
mmr_docs = mmr_retriever.invoke("batting performance")
print(f"MMR → {len(mmr_docs)} diverse chunks retrieved")

MMR → 4 diverse chunks retrieved


In [26]:
# ── format_docs — converts list of Documents to one context string ──
# This is called INSIDE the RAG chain to prepare the {context} for the prompt
def format_docs(docs):
    return "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'unknown')}]\n{doc.page_content}"
        for doc in docs
    )

# Test it
test_context = format_docs(retriever.invoke("best bowler"))
print(test_context[:600])

[Source: C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\match_summary.txt]
=== CSK BOWLING (vs DC) ===
Akeal Hosein: 4 overs, 0 maiden, 19 runs, 1 wicket, Econ: 4.75
Mukesh Choudhary: 4 overs, 0 maiden, 31 runs, 1 wicket, Econ: 7.75
Anshul Kamboj: 4 overs, 0 maiden, 49 runs, 0 wickets, Econ: 12.25
Noor Ahmad: 3 overs, 0 maiden, 22 runs, 2 wickets, Econ: 7.33
Gurjapneet Singh: 4 overs, 0 maiden, 29 runs, 1 wicket, Econ: 7.25
Jamie Overton: 1 over, 0 maiden, 5 runs, 1 wicket, Econ: 5.00

=== POST MATCH QUOTES ===

[Source: C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\


---
# ══════════════════════════════════════════════════
# FULL RAG CHAIN (LCEL)
# ══════════════════════════════════════════════════

## Theory : What is RAG?

**RAG = Retrieval-Augmented Generation**

A plain LLM like LLaMA only knows what it was trained on (data up to a cutoff date).  
It has **zero knowledge** of your CSV files, your match summary, or the live IPL table.

**RAG bridges this gap:**
1. Takes the user's question
2. **Retrieves** the most relevant chunks from YOUR data
3. Injects those chunks as **context** into the prompt
4. Lets the LLM **generate** an answer grounded in YOUR data

```
User: "Who scored the most runs?"
       │
       ├──► Retriever ──► finds "Rohit: 85, Kohli: 72..." from batting_scorecard.csv
       │                               │
       └───────────────────────────────▼
                     Prompt:  SYSTEM = "Use this context: Rohit: 85, Kohli: 72..."
                              HUMAN  = "Who scored the most runs?"
                                       │
                                       ▼
                                 Groq LLaMA 3.3
                                       │
                                       ▼
                          "Rohit Sharma scored the most with 85 runs."
```

---
## 🔑 Why LCEL Instead of Old Chains?

Old LangChain (deprecated, DO NOT USE):
```python
# Old — deprecated in 2025
chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)
```

New LCEL (2026: use this):
```python
# New — transparent, composable, production-ready
chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()}
         | rag_prompt | llm | StrOutputParser())
```

LCEL benefits:
- Every step is visible and debuggable
- Easily add/remove steps
- Streaming works out of the box
- Parallel execution supported natively

In [27]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

# ════════════════════════════════════════════════════
# THE RAG CHAIN — How it works step by step:
#
# RunnableParallel runs TWO branches simultaneously:
#   Branch 1 "context"  : question → retriever → format_docs → string
#   Branch 2 "question" : question passes through unchanged
#
# Both outputs feed into rag_prompt as {context} and {question}
# Then: rag_prompt → llm → StrOutputParser
# ════════════════════════════════════════════════════

rag_chain = (
    RunnableParallel({
        "context" : retriever | format_docs,
        "question": RunnablePassthrough()
    })
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG Chain ready — pipeline: Retriever → Prompt → LLM → Parser")

RAG Chain ready — pipeline: Retriever → Prompt → LLM → Parser


In [28]:
# ── Test 1 — Batting Question ──
q = "Who scored the highest runs in the batting scorecard?"
print(f"Q: {q}")
print(f"A: {rag_chain.invoke(q)}")

Q: Who scored the highest runs in the batting scorecard?
A: Sameer Rizvi scored the highest runs with 40* off 24 balls.


In [29]:
# ── Test 1 — Batting Question ──
q = "performance of noor ahmed"
print(f"Q: {q}")
print(f"A: {rag_chain.invoke(q)}")

Q: performance of noor ahmed
A: Noor Ahmad's performance: 
- Overs: 3
- Maidens: 0
- Runs: 22
- Wickets: 2
- Economy: 7.33 

He also got 2 batters from Delhi Capitals out, namely Nitish Rana and Karun Nair.


In [30]:
print("🏏 Cricket RAG Assistant — Ask anything about the match data!")
print("Type 'quit' to exit\n")

while True:
    question = input("You: ").strip()
    if question.lower() in ["quit", "exit", "q"]:
        print("Bye!")
        break
    if not question:
        continue
    answer = rag_chain.invoke(question)
    print(f"\n🤖 Bot: {answer}\n")
    print("-" * 50)

🏏 Cricket RAG Assistant — Ask anything about the match data!
Type 'quit' to exit

Bye!


# Cosine Similarity vs Centroid-Based Vector Search vs Retrieval

---

## 1. Cosine Similarity
- **What it is:** A mathematical measure of how close two vectors are in direction.  
- **How it works:** Given a query vector and a stored vector, cosine similarity calculates the angle between them. Smaller angle → higher similarity.  
- **Use case:** Directly compares one query against one document chunk.  
- **Limitation:** If you have millions of vectors, computing cosine similarity against all of them becomes too slow.

**Example:**  
Query: "Rohit Sharma scored a century"  
Stored chunk: "Batsman made 100 runs in IPL"  
Cosine similarity ≈ 0.9 → very similar.

---

## 2. Centroid-Based Vector Search (Indexing / ANN)
- **What it is:** A way to make similarity search faster by grouping vectors into clusters.  
- **How it works:**  
  1. Vectors are organized into clusters, each with a centroid (average vector).  
  2. The query is first compared to centroids (fast).  
  3. Only the nearest cluster is searched in detail using cosine similarity.  
- **Use case:** Large-scale datasets (millions of vectors).  
- **Tradeoff:** It’s “approximate” — you may miss the absolute best match, but you gain huge speed improvements.

**Example:**  
Database: 1 million news articles clustered into 1,000 groups.  
Query: "Who won the T20 World Cup?"  
Step 1: Compare query with 1,000 centroids → nearest is cricket cluster.  
Step 2: Compare query only with ~1,000 cricket articles.  
Result: ~2,000 comparisons instead of 1,000,000.

---

## 3. Retriever
- **What it is:** A wrapper/interface around the vector store that makes it usable in pipelines (like RAG).  
- **How it works:**  
  - Accepts a query.  
  - Calls the vector store (which internally uses cosine similarity + indexing).  
  - Returns the top‑K most relevant chunks.  
- **Use case:** Needed when integrating with LLMs, because retrievers are “runnables” that can plug into chains.  
- **Benefit:** Abstracts away the complexity — you just call `.invoke(query)` and get results.

**Example:**  
User asks: "How many wickets did Noor Ahmad take?"  
Retriever:  
1. Embeds the query.  
2. Uses vector store (FAISS) to run similarity search.  
3. Returns top‑3 relevant IPL chunks.  
4. Passes them to the LLM for answering.

---

## Key Differences
- **Cosine similarity** → the actual math for comparing two vectors.  
- **Centroid-based vector search (ANN)** → optimization technique to avoid brute-force cosine similarity across all vectors.  
- **Retriever** → the interface that connects queries to the vector store, making it usable in RAG pipelines.

---

## Takeaway
Think of it like this:  
- **Cosine similarity** is the *ruler* that measures closeness.  
- **Centroid/ANN indexing** is the *shortcut* that avoids measuring everything.  
- **Retriever** is the *messenger* that takes your query, uses the shortcut + ruler, and delivers the most relevant chunks to the LLM.


# Cosine Similarity vs Vector Store Indexing vs Retriever

---

## 1. Cosine Similarity
- **Definition:** A mathematical formula that measures how close two vectors are in direction.  
- **Where it’s used:**  
  - Inside the **vector store** when comparing a query vector against stored vectors.  
  - Inside the **retriever**, because the retriever calls the vector store and relies on cosine similarity to rank results.  
- **Key Point:** Cosine similarity is the *metric* used to judge closeness between embeddings.

**Example:**  
Query vector for “Rohit Sharma century” compared with stored vector for “Batsman scored 100 runs” → cosine similarity ≈ 0.9 (high match).

---

## 2. Vector Store Indexing (Centroids / ANN)
- **Definition:** An optimization technique to avoid brute-force cosine similarity across millions of vectors.  
- **How it works:**  
  - Vectors are grouped into clusters with centroids.  
  - Query is first compared to centroids (fast).  
  - Only the nearest cluster is searched in detail using cosine similarity.  
- **Key Point:** Indexing reduces the number of cosine similarity calculations needed, making search scalable.

**Example:**  
Instead of comparing against 1,000,000 articles, the query first checks 1,000 centroids, then only searches ~1,000 articles in the nearest cluster.  
Total ≈ 2,000 comparisons instead of 1,000,000.

---

## 3. Retriever
- **Definition:** A wrapper around the vector store that makes it usable in pipelines (like RAG).  
- **How it works:**  
  - Accepts a query.  
  - Calls the vector store (which internally uses cosine similarity + indexing).  
  - Returns the top‑K most relevant chunks.  
- **Key Point:** The retriever doesn’t invent a new metric — it simply exposes vector store search (which uses cosine similarity) in a pipeline-friendly way.

**Example:**  
User asks: “How many wickets did Noor Ahmad take?”  
Retriever → embeds query → vector store runs cosine similarity search → returns top‑3 relevant IPL chunks → LLM answers.

---

## Final Clarification
- **Cosine similarity** is the *mathematical measure* used everywhere to compare vectors.  
- **Indexing/ANN** is the *optimization layer* that reduces how many cosine similarity checks are needed.  
- **Retriever** is the *interface* that plugs vector store search (with cosine similarity + indexing) into RAG pipelines.

So yes — cosine similarity is used in both **vector store search** and **retriever**, but indexing decides *how many* cosine similarity calculations are performed.
